# Загрузка тиковых данных BTC/USDT с Bybit

Исторические тиковые данные скачиваются с публичного хранилища Bybit (`public.bybit.com/trading`) за период 2020–2026. Каждый день — отдельный файл, загрузка параллельная через `ThreadPoolExecutor`. Сырой CSV конвертируется в Parquet с Snappy-сжатием: итоговый размер ~6x меньше исходного.

Схема одного тика после парсинга:

| Поле | Тип | Описание |
|---|---|---|
| `dt` | datetime64[UTC] | Время сделки |
| `price` | float64 | Цена |
| `size` | float32 | Объём в BTC |
| `dollar_value` | float64 | price × size |
| `b_t` | int8 | Направление: Buy=+1, Sell=−1 |
| `side` | category | Buy / Sell |
| `tickDirection` | category | Направление тика |

Параметры конфигурации:

| Параметр | Описание |
|---|---|
| `START_DATE` / `END_DATE` | Период загрузки |
| `MAX_WORKERS` | Число параллельных потоков `ThreadPoolExecutor` |
| `DELAY` | Задержка между отправкой задач в пул — снижает нагрузку на сервер Bybit и уменьшает число сброшенных соединений |

In [ ]:
import requests
import gzip
import shutil
import os
import time
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

SYMBOL      = "BTCUSDT"
BASE_URL    = "https://public.bybit.com/trading"
TICKS_DIR   = Path("data/ticks/BTCUSDT")
TICKS_DIR.mkdir(parents=True, exist_ok=True)

START_DATE  = date(2020, 3, 25)
END_DATE    = date(2026, 1, 1)
MAX_WORKERS = 8
DELAY       = 0.5 

## Загрузка

Каждый день скачивается независимо. Если файл уже есть — пропускается. При сетевой ошибке временные файлы удаляются, день помечается как `fail` — можно перезапустить ячейку: уже скачанные дни будут пропущены, попытка повторится только для недостающих.

`date_range` — генератор дат от `START_DATE` до `END_DATE`. `download_day` — основная единица работы: скачивает `.csv.gz`, распаковывает, парсит в DataFrame, добавляет поля `dt`, `b_t`, `dollar_value`, сохраняет в Parquet и удаляет временные файлы. Возвращает статус `ok` / `skip` / `fail`.

In [10]:
def date_range(start: date, end: date):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)


def download_day(d: date) -> tuple[date, str, float]:
    date_str     = d.strftime("%Y-%m-%d")
    parquet_path = TICKS_DIR / f"{SYMBOL}{date_str}.parquet"
    gz_path      = TICKS_DIR / f"{SYMBOL}{date_str}.csv.gz"
    csv_path     = TICKS_DIR / f"{SYMBOL}{date_str}.csv"

    if parquet_path.exists():
        size_mb = os.path.getsize(parquet_path) / 1024 / 1024
        return d, "skip", size_mb

    url = f"{BASE_URL}/{SYMBOL}/{SYMBOL}{date_str}.csv.gz"

    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(gz_path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    except Exception as e:
        for p in [gz_path, csv_path]:
            if p.exists(): p.unlink()
        return d, f"fail:{e}", 0.0

    try:
        with gzip.open(gz_path, "rb") as f_in, open(csv_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_path.unlink()
    except Exception as e:
        for p in [gz_path, csv_path]:
            if p.exists(): p.unlink()
        return d, f"fail:{e}", 0.0

    try:
        df = pd.read_csv(
            csv_path,
            dtype={
                "timestamp":       "float64",
                "symbol":          "category",
                "side":            "category",
                "size":            "float32",
                "price":           "float64",
                "tickDirection":   "category",
                "trdMatchID":      "str",
                "grossValue":      "float64",
                "homeNotional":    "float32",
                "foreignNotional": "float64",
            }
        )
        df["dt"]           = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["b_t"]          = df["side"].map({"Buy": 1, "Sell": -1}).astype("int8")
        df["dollar_value"] = df["price"] * df["size"]
        df = df.sort_values("dt").reset_index(drop=True)
        csv_path.unlink()
        df.to_parquet(parquet_path, compression="snappy", index=False)
        size_mb = os.path.getsize(parquet_path) / 1024 / 1024
        return d, "ok", size_mb
    except Exception as e:
        for p in [csv_path, parquet_path]:
            if p.exists(): p.unlink()
        return d, f"fail:{e}", 0.0

## Результат

Все 2109 файлов уже присутствуют на диске — загрузка пропущена. Итоговый размер датасета **122.75 GB** в формате Parquet со Snappy-сжатием. Исходные CSV весили бы порядка ~700 GB.

In [11]:
all_dates = list(date_range(START_DATE, END_DATE))
total     = len(all_dates)

print(f"Период:  {START_DATE} → {END_DATE}  ({total} дней)")
print(f"Потоков: {MAX_WORKERS}")
print(f"Папка:   {TICKS_DIR.resolve()}\n")

ok = skipped = failed = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {}
    for d in all_dates:
        fut = executor.submit(download_day, d)
        futures[fut] = d
        time.sleep(DELAY)

    for fut in as_completed(futures):
        d, status, size_mb = fut.result()

        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed += 1
            print(f"  {d}  {status}")

        done = ok + skipped + failed
        if done % 100 == 0:
            print(f"  [{done}/{total}]  скачано: {ok}  пропущено: {skipped}  ошибок: {failed}")

total_mb = sum(f.stat().st_size for f in TICKS_DIR.glob("*.parquet")) / 1024 / 1024
print(f"\nСкачано: {ok}  |  Пропущено: {skipped}  |  Ошибок: {failed}")
print(f"Размер:  {total_mb:.1f} MB ({total_mb/1024:.2f} GB)")

Период:  2020-03-25 → 2026-01-01  (2109 дней)
Потоков: 8
Папка:   C:\Users\User\Documents\quantum_break2\data\ticks\BTCUSDT

  [100/2109]  скачано: 0  пропущено: 100  ошибок: 0
  [200/2109]  скачано: 0  пропущено: 200  ошибок: 0
  [300/2109]  скачано: 0  пропущено: 300  ошибок: 0
  [400/2109]  скачано: 0  пропущено: 400  ошибок: 0
  [500/2109]  скачано: 0  пропущено: 500  ошибок: 0
  [600/2109]  скачано: 0  пропущено: 600  ошибок: 0
  [700/2109]  скачано: 0  пропущено: 700  ошибок: 0
  [800/2109]  скачано: 0  пропущено: 800  ошибок: 0
  [900/2109]  скачано: 0  пропущено: 900  ошибок: 0
  [1000/2109]  скачано: 0  пропущено: 1000  ошибок: 0
  [1100/2109]  скачано: 0  пропущено: 1100  ошибок: 0
  [1200/2109]  скачано: 0  пропущено: 1200  ошибок: 0
  [1300/2109]  скачано: 0  пропущено: 1300  ошибок: 0
  [1400/2109]  скачано: 0  пропущено: 1400  ошибок: 0
  [1500/2109]  скачано: 0  пропущено: 1500  ошибок: 0
  [1600/2109]  скачано: 0  пропущено: 1600  ошибок: 0
  [1700/2109]  скачано: 0  пр

## Артефакты данных

В ходе валидации обнаружено три класса проблем — все объясняются особенностями площадки на раннем этапе работы:

**Гэпы > 10 минут** (2020-03-25, 2020-03-26 и ряд дат в июне) — технические перерывы в матчинге Bybit. Биржа запустила BTCUSDT perpetual в марте 2020 и в первые месяцы периодически уходила на технические остановки. При построении dollar bars это просто даст пустой период.

**Нулевые `size`** (2020-06-28, 06-29, 07-05, 07-07) — по одному тику с `size=0`, артефакт выгрузки. Фиксируется при построении баров: `df = df[df["size"] > 0]`.

**Дубли `trdMatchID`** (2020-11-24, 2022-02-28) — дублированные записи одной сделки. Фиксируется: `df = df.drop_duplicates("trdMatchID")`.

In [ ]:
files = sorted(TICKS_DIR.glob("BTCUSDT*.parquet"))

fixed = 0

for f in files:
    df           = pd.read_parquet(f)
    original_len = len(df)
    changed      = False

    mask_zero = df["size"] <= 0
    if mask_zero.any():
        df      = df[~mask_zero]
        changed = True

    mask_dupes = df.duplicated("trdMatchID")
    if mask_dupes.any():
        df      = df.drop_duplicates("trdMatchID")
        changed = True

    if changed:
        removed = original_len - len(df)
        df.to_parquet(f, compression="snappy", index=False)
        
        fixed += 1



## Итог

Тиковые данные BTC/USDT за 2020–2026 собраны и очищены — **2109 дней, 122.75 GB**. Каждая запись отражает реальную сделку на бирже: цену, объём, направление и время с точностью до миллисекунды. Это наиболее подробное представление рыночной микроструктуры, доступное публично.

Следующий шаг — построение информационно-эффективных баров: dollar bars, volume bars, tick bars.